In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from catboost import CatBoostClassifier
import time

train_df = pd.read_csv('../data/processed/train_fe.csv')
test_df = pd.read_csv('../data/processed/test_fe.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

categorical_cols = ['diet_type', 'stress_level', 'sleep_quality', 
                     'physical_activity_level', 'smoking_alcohol', 'gender',
                     'stress_activity_combo', 'sleep_duration_bin', 'bmi_category']

for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')
    test_df[col] = test_df[col].astype('category')

target_map = {'at-risk': 0, 'unhealthy': 1, 'fit': 2}
target_map_inv = {v: k for k, v in target_map.items()}
train_df['target_encoded'] = train_df['health_condition'].map(target_map)

feature_cols = [c for c in train_df.columns if c not in ['id', 'health_condition', 'target_encoded']]

X = train_df[feature_cols]
y = train_df['target_encoded']
X_test = test_df[feature_cols]

print("\nFeature columns:", feature_cols)
print("\nTarget distribution:")
print(y.value_counts())

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

lgb_oof_preds = np.zeros(len(X))
lgb_test_preds = np.zeros((len(X_test), N_FOLDS))
lgb_fold_scores = []

lgb_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'class_weight': 'balanced',
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'n_estimators': 1000,
    'random_state': 42,
    'verbosity': -1
}

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=categorical_cols,
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    val_preds = model.predict(X_val)
    lgb_oof_preds[val_idx] = val_preds
    lgb_test_preds[:, fold] = model.predict(X_test)
    
    fold_score = balanced_accuracy_score(y_val, val_preds)
    lgb_fold_scores.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - Balanced Accuracy: {fold_score:.4f}")

lgb_cv_score = balanced_accuracy_score(y, lgb_oof_preds)
print(f"\n{'='*50}")
print(f"LightGBM Overall OOF Balanced Accuracy: {lgb_cv_score:.4f}")
print(f"Mean fold score: {np.mean(lgb_fold_scores):.4f} (+/- {np.std(lgb_fold_scores):.4f})")
print(f"Time taken: {time.time()-start_time:.1f}s")

In [ ]:
cat_oof_preds = np.zeros(len(X))
cat_test_preds = np.zeros((len(X_test), N_FOLDS))
cat_fold_scores = []

cat_feature_indices = [X.columns.get_loc(c) for c in categorical_cols]

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    for col in categorical_cols:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)
    
    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function='MultiClass',
        auto_class_weights='Balanced',
        cat_features=cat_feature_indices,
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50
    )
    
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val)
    )
    
    val_preds = model.predict(X_val).flatten()
    cat_oof_preds[val_idx] = val_preds
    
    X_test_cat = X_test.copy()
    for col in categorical_cols:
        X_test_cat[col] = X_test_cat[col].astype(str)
    cat_test_preds[:, fold] = model.predict(X_test_cat).flatten()
    
    fold_score = balanced_accuracy_score(y_val, val_preds)
    cat_fold_scores.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - Balanced Accuracy: {fold_score:.4f}")

cat_cv_score = balanced_accuracy_score(y, cat_oof_preds)
print(f"\n{'='*50}")
print(f"CatBoost Overall OOF Balanced Accuracy: {cat_cv_score:.4f}")
print(f"Mean fold score: {np.mean(cat_fold_scores):.4f} (+/- {np.std(cat_fold_scores):.4f})")
print(f"Time taken: {time.time()-start_time:.1f}s")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y, cat_oof_preds)
cm_labels = ['at-risk', 'unhealthy', 'fit']

print("Confusion Matrix (CatBoost OOF):")
cm_df = pd.DataFrame(cm, index=[f'true_{l}' for l in cm_labels], 
                      columns=[f'pred_{l}' for l in cm_labels])
print(cm_df)

print("\nNormalized (row %):")
cm_norm = cm / cm.sum(axis=1, keepdims=True) * 100
cm_norm_df = pd.DataFrame(cm_norm.round(2), index=[f'true_{l}' for l in cm_labels], 
                            columns=[f'pred_{l}' for l in cm_labels])
print(cm_norm_df)

print("\nClassification Report:")
print(classification_report(y, cat_oof_preds, target_names=cm_labels, digits=4))

In [ ]:
lgb_oof_probs = np.zeros((len(X), 3))
lgb_test_probs = np.zeros((len(X_test), 3))

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=categorical_cols,
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    lgb_oof_probs[val_idx] = model.predict_proba(X_val)
    lgb_test_probs += model.predict_proba(X_test) / N_FOLDS
    
    fold_hard_preds = np.argmax(model.predict_proba(X_val), axis=1)
    print(f"Fold {fold+1}/{N_FOLDS} - Balanced Accuracy (default argmax): {balanced_accuracy_score(y_val, fold_hard_preds):.4f}")

print(f"\nTime: {time.time()-start_time:.1f}s")

baseline_hard_preds = np.argmax(lgb_oof_probs, axis=1)
baseline_score = balanced_accuracy_score(y, baseline_hard_preds)
print(f"\nLightGBM baseline (argmax) Balanced Accuracy: {baseline_score:.4f}")

In [ ]:
from scipy.optimize import minimize

def apply_weights_and_predict(probs, weights):
    weighted_probs = probs * weights
    return np.argmax(weighted_probs, axis=1)

def negative_balanced_accuracy(weights, probs, y_true):
    full_weights = np.array([1.0, weights[0], weights[1]])
    preds = apply_weights_and_predict(probs, full_weights)
    score = balanced_accuracy_score(y_true, preds)
    return -score

result = minimize(
    negative_balanced_accuracy, 
    x0=[1.0, 1.0], 
    args=(lgb_oof_probs, y),
    method='Nelder-Mead',
    bounds=[(0.3, 3.0), (0.3, 3.0)],
    options={'xatol': 0.01, 'fatol': 0.0001, 'maxiter': 100}
)

best_weights = np.array([1.0, result.x[0], result.x[1]])
print("Best weights [at-risk, unhealthy, fit]:", best_weights)
print(f"Optimized Balanced Accuracy: {-result.fun:.4f}")
print(f"Baseline (no weights) Balanced Accuracy: {baseline_score:.4f}")
print(f"Improvement: {(-result.fun - baseline_score)*100:.2f} percentage points")

optimized_preds = apply_weights_and_predict(lgb_oof_probs, best_weights)
cm_opt = confusion_matrix(y, optimized_preds)
cm_norm_opt = cm_opt / cm_opt.sum(axis=1, keepdims=True) * 100
print("\nOptimized Confusion Matrix (row %):")
print(pd.DataFrame(cm_norm_opt.round(2), index=[f'true_{l}' for l in cm_labels], columns=[f'pred_{l}' for l in cm_labels]))

print("\nOptimized Classification Report:")
print(classification_report(y, optimized_preds, target_names=cm_labels, digits=4))

In [ ]:
print("Variables available:")
print("cat_oof_preds exists:", 'cat_oof_preds' in dir())
print("cat_test_preds exists:", 'cat_test_preds' in dir())

if 'cat_oof_preds' in dir():
    print("\ncat_oof_preds shape:", cat_oof_preds.shape)
    print("cat_oof_preds sample values:", cat_oof_preds[:10])
    print("Is this probabilities (should be 2D, values 0-1) or hard labels (1D, integers)?")
    print("Shape is 1D:", cat_oof_preds.ndim == 1)

In [ ]:
import optuna

def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'class_weight': 'balanced',
        'metric': 'multi_logloss',
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'n_estimators': 1000,
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': 5,
        'random_state': 42,
        'verbosity': -1
    }
    
    skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in skf_tune.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            categorical_feature=categorical_cols,
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )
        preds = model.predict(X_val)
        scores.append(balanced_accuracy_score(y_val, preds))
    
    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

In [ ]:
best_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'class_weight': 'balanced',
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'n_estimators': 1500,
    'random_state': 42,
    'verbosity': -1,
    **study.best_params
}

final_oof_probs = np.zeros((len(X), 3))
final_test_probs = np.zeros((len(X_test), 3))
final_fold_scores = []

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**best_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=categorical_cols,
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    val_probs = model.predict_proba(X_val)
    final_oof_probs[val_idx] = val_probs
    final_test_probs += model.predict_proba(X_test) / N_FOLDS

    fold_preds = np.argmax(val_probs, axis=1)
    fold_score = balanced_accuracy_score(y_val, fold_preds)
    final_fold_scores.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - Balanced Accuracy: {fold_score:.4f}")

final_hard_preds = np.argmax(final_oof_probs, axis=1)
final_cv_score = balanced_accuracy_score(y, final_hard_preds)

print(f"\n{'='*50}")
print(f"Tuned LightGBM Overall OOF Balanced Accuracy: {final_cv_score:.4f}")
print(f"Mean fold score: {np.mean(final_fold_scores):.4f} (+/- {np.std(final_fold_scores):.4f})")
print(f"Time: {time.time()-start_time:.1f}s")

In [ ]:
cat_oof_probs = np.zeros((len(X), 3))
cat_test_probs = np.zeros((len(X_test), 3))
cat_fold_scores_v2 = []

start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    for col in categorical_cols:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function='MultiClass',
        auto_class_weights='Balanced',
        cat_features=cat_feature_indices,
        random_seed=42,
        verbose=False,
        early_stopping_rounds=50
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val))

    val_probs = model.predict_proba(X_val)
    cat_oof_probs[val_idx] = val_probs

    X_test_cat = X_test.copy()
    for col in categorical_cols:
        X_test_cat[col] = X_test_cat[col].astype(str)
    cat_test_probs += model.predict_proba(X_test_cat) / N_FOLDS

    fold_preds = np.argmax(val_probs, axis=1)
    fold_score = balanced_accuracy_score(y_val, fold_preds)
    cat_fold_scores_v2.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - Balanced Accuracy: {fold_score:.4f}")

cat_hard_preds = np.argmax(cat_oof_probs, axis=1)
cat_cv_score_v2 = balanced_accuracy_score(y, cat_hard_preds)
print(f"\nCatBoost Overall OOF Balanced Accuracy: {cat_cv_score_v2:.4f}")
print(f"Time: {time.time()-start_time:.1f}s")

In [ ]:
from scipy.optimize import minimize

def blend_and_score(w, probs1, probs2, y_true):
    blended = w * probs1 + (1 - w) * probs2
    preds = np.argmax(blended, axis=1)
    return balanced_accuracy_score(y_true, preds)

weights = np.arange(0, 1.01, 0.05)
blend_scores = [blend_and_score(w, final_oof_probs, cat_oof_probs, y) for w in weights]

best_idx = np.argmax(blend_scores)
best_w = weights[best_idx]

print("Weight (LightGBM share) -> Balanced Accuracy:")
for w, s in zip(weights, blend_scores):
    marker = " <-- best" if w == best_w else ""
    print(f"{w:.2f}: {s:.4f}{marker}")

print(f"\nBest blend weight (LightGBM): {best_w:.2f}")
print(f"Best blend Balanced Accuracy: {blend_scores[best_idx]:.4f}")
print(f"\nCompare:")
print(f"LightGBM alone: {final_cv_score:.4f}")
print(f"CatBoost alone: {cat_cv_score_v2:.4f}")
print(f"Blend: {blend_scores[best_idx]:.4f}")

In [ ]:
blended_probs = best_w * final_oof_probs + (1 - best_w) * cat_oof_probs

def apply_weights_and_predict(probs, weights):
    weighted_probs = probs * weights
    return np.argmax(weighted_probs, axis=1)

def negative_balanced_accuracy_blend(weights, probs, y_true):
    full_weights = np.array([1.0, weights[0], weights[1]])
    preds = apply_weights_and_predict(probs, full_weights)
    return -balanced_accuracy_score(y_true, preds)

result_blend = minimize(
    negative_balanced_accuracy_blend,
    x0=[1.0, 1.0],
    args=(blended_probs, y),
    method='Nelder-Mead',
    bounds=[(0.3, 3.0), (0.3, 3.0)],
    options={'xatol': 0.01, 'fatol': 0.0001, 'maxiter': 100}
)

best_class_weights = np.array([1.0, result_blend.x[0], result_blend.x[1]])
print("Best class weights [at-risk, unhealthy, fit]:", best_class_weights)
print(f"Final optimized Balanced Accuracy: {-result_blend.fun:.4f}")

final_optimized_preds = apply_weights_and_predict(blended_probs, best_class_weights)
cm_final = confusion_matrix(y, final_optimized_preds)
cm_norm_final = cm_final / cm_final.sum(axis=1, keepdims=True) * 100

print("\nFinal Confusion Matrix (row %):")
print(pd.DataFrame(cm_norm_final.round(2), index=[f'true_{l}' for l in cm_labels], columns=[f'pred_{l}' for l in cm_labels]))

print("\nFinal Classification Report:")
print(classification_report(y, final_optimized_preds, target_names=cm_labels, digits=4))

In [ ]:
blended_test_probs = best_w * final_test_probs + (1 - best_w) * cat_test_probs
final_test_preds = apply_weights_and_predict(blended_test_probs, best_class_weights)

print("final_test_preds shape:", final_test_preds.shape)
print("Sample:", final_test_preds[:10])

In [ ]:
test_ids = test_df['id']

submission = pd.DataFrame({
    'id': test_ids,
    'health_condition': [target_map_inv[p] for p in final_test_preds]
})

print("Submission shape:", submission.shape)
print("\nPrediction distribution:")
print(submission['health_condition'].value_counts(normalize=True) * 100)

submission.to_csv('../data/processed/submission.csv', index=False)
print("\nSaved submission.csv")
print(submission.head())

In [ ]:
import numpy as np
import os

os.makedirs('../data/processed/oof', exist_ok=True)

np.save('../data/processed/oof/lgb_v1_oof_probs.npy', final_oof_probs)
np.save('../data/processed/oof/lgb_v1_test_probs.npy', final_test_probs)
np.save('../data/processed/oof/cat_oof_probs.npy', cat_oof_probs)
np.save('../data/processed/oof/cat_test_probs.npy', cat_test_probs)

print("Saved all OOF/test probability arrays")

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

dummy_oof_preds = np.zeros(len(X))
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train, y_train)
    dummy_oof_preds[val_idx] = dummy.predict(X_val)

dummy_cv_score = balanced_accuracy_score(y, dummy_oof_preds)
print(f"Dummy (majority class) Balanced Accuracy: {dummy_cv_score:.4f}")

numeric_feature_cols = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                         'step_count', 'exercise_duration', 'water_intake',
                         'activity_score', 'calorie_per_step']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_feature_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

lr_oof_preds = np.zeros(len(X))
lr_fold_scores = []
start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    lr_pipe = Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))
    ])
    lr_pipe.fit(X_train, y_train)
    val_preds = lr_pipe.predict(X_val)
    lr_oof_preds[val_idx] = val_preds

    fold_score = balanced_accuracy_score(y_val, val_preds)
    lr_fold_scores.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - LogisticRegression Balanced Accuracy: {fold_score:.4f}")

lr_cv_score = balanced_accuracy_score(y, lr_oof_preds)
print(f"\nLogisticRegression Overall OOF Balanced Accuracy: {lr_cv_score:.4f}")
print(f"Time: {time.time()-start_time:.1f}s")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder

X_rf = X.copy()
X_test_rf = X_test.copy()

ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_rf[categorical_cols] = ordinal_enc.fit_transform(X_rf[categorical_cols].astype(str))
X_test_rf[categorical_cols] = ordinal_enc.transform(X_test_rf[categorical_cols].astype(str))

rf_oof_probs = np.zeros((len(X), 3))
rf_test_probs = np.zeros((len(X_test), 3))
rf_fold_scores = []
start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_rf, y)):
    X_train, X_val = X_rf.iloc[train_idx], X_rf.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    rf_model = RandomForestClassifier(
        n_estimators=500,
        max_depth=16,
        min_samples_leaf=5,
        class_weight='balanced',
        n_jobs=-1,
        random_state=42
    )
    rf_model.fit(X_train, y_train)

    val_probs = rf_model.predict_proba(X_val)
    rf_oof_probs[val_idx] = val_probs
    rf_test_probs += rf_model.predict_proba(X_test_rf) / N_FOLDS

    fold_preds = np.argmax(val_probs, axis=1)
    fold_score = balanced_accuracy_score(y_val, fold_preds)
    rf_fold_scores.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - RandomForest Balanced Accuracy: {fold_score:.4f}")

rf_hard_preds = np.argmax(rf_oof_probs, axis=1)
rf_cv_score = balanced_accuracy_score(y, rf_hard_preds)
print(f"\nRandomForest Overall OOF Balanced Accuracy: {rf_cv_score:.4f}")
print(f"Time: {time.time()-start_time:.1f}s")

In [ ]:
import xgboost as xgb

X_xgb = X.copy()
X_test_xgb = X_test.copy()
for col in categorical_cols:
    X_xgb[col] = X_xgb[col].astype('category')
    X_test_xgb[col] = X_test_xgb[col].astype('category')

class_counts = y.value_counts()
class_weight_map = (class_counts.sum() / class_counts).to_dict()

xgb_oof_probs = np.zeros((len(X), 3))
xgb_test_probs = np.zeros((len(X_test), 3))
xgb_fold_scores = []
start_time = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(X_xgb, y)):
    X_train, X_val = X_xgb.iloc[train_idx], X_xgb.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    sw_train = y_train.map(class_weight_map).values

    xgb_model = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=8,
        enable_categorical=True,
        tree_method='hist',
        eval_metric='mlogloss',
        early_stopping_rounds=50,
        random_state=42
    )
    xgb_model.fit(
        X_train, y_train,
        sample_weight=sw_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    val_probs = xgb_model.predict_proba(X_val)
    xgb_oof_probs[val_idx] = val_probs
    xgb_test_probs += xgb_model.predict_proba(X_test_xgb) / N_FOLDS

    fold_preds = np.argmax(val_probs, axis=1)
    fold_score = balanced_accuracy_score(y_val, fold_preds)
    xgb_fold_scores.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - XGBoost Balanced Accuracy: {fold_score:.4f}")

xgb_hard_preds = np.argmax(xgb_oof_probs, axis=1)
xgb_cv_score = balanced_accuracy_score(y, xgb_hard_preds)
print(f"\nXGBoost Overall OOF Balanced Accuracy: {xgb_cv_score:.4f}")
print(f"Time: {time.time()-start_time:.1f}s")

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Dummy (majority)', 'Logistic Regression', 'Random Forest', 'XGBoost',
              'LightGBM (tuned)', 'CatBoost'],
    'OOF Balanced Accuracy': [dummy_cv_score, lr_cv_score, rf_cv_score, xgb_cv_score,
                               final_cv_score, cat_cv_score_v2]
}).sort_values('OOF Balanced Accuracy', ascending=False).reset_index(drop=True)

comparison_df['Balanced Accuracy'] = comparison_df['OOF Balanced Accuracy'].map('{:.4f}'.format)
print("Model Comparison (sorted by OOF Balanced Accuracy):\n")
print(comparison_df[['Model', 'Balanced Accuracy']].to_string(index=False))

In [ ]:
meta_X = np.hstack([final_oof_probs, cat_oof_probs, rf_oof_probs, xgb_oof_probs])
meta_X_test = np.hstack([final_test_probs, cat_test_probs, rf_test_probs, xgb_test_probs])

meta_oof_preds = np.zeros(len(X))
meta_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(meta_X, y)):
    meta_train, meta_val = meta_X[train_idx], meta_X[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    meta_model = LogisticRegression(max_iter=1000)
    meta_model.fit(meta_train, y_train)
    val_preds = meta_model.predict(meta_val)
    meta_oof_preds[val_idx] = val_preds

    fold_score = balanced_accuracy_score(y_val, val_preds)
    meta_fold_scores.append(fold_score)
    print(f"Fold {fold+1}/{N_FOLDS} - Stacked Meta-Model Balanced Accuracy: {fold_score:.4f}")

meta_cv_score = balanced_accuracy_score(y, meta_oof_preds)
print(f"\nStacked Meta-Model Overall OOF Balanced Accuracy: {meta_cv_score:.4f}")
print(f"\nCompare:")
print(f"  Manual weighted blend (LGB+Cat): {blend_scores[best_idx]:.4f}")
print(f"  Stacked ensemble (all 4 models): {meta_cv_score:.4f}")

final_meta_model = LogisticRegression(max_iter=1000).fit(meta_X, y)
final_stacked_test_preds = final_meta_model.predict(meta_X_test)

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

os.makedirs('../reports', exist_ok=True)

class_names = [target_map_inv[i] for i in range(3)]

cm_counts = confusion_matrix(y, final_hard_preds)
cm_normalized = confusion_matrix(y, final_hard_preds, normalize='true')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm_counts, annot=True, fmt=',d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
axes[0].set_title('Confusion Matrix (raw counts)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1], cbar=False)
axes[1].set_title('Confusion Matrix (row-normalized = recall per class)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../reports/confusion_matrix_lightgbm.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y, final_hard_preds, target_names=class_names, digits=4))

In [ ]:
import joblib
import lightgbm as lgb

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y, categorical_feature=categorical_cols)

import os
os.makedirs('../models', exist_ok=True)

joblib.dump({
    'model': final_model,
    'feature_cols': feature_cols,
    'categorical_cols': categorical_cols,
    'target_map_inv': target_map_inv,
}, '../models/health_condition_model.pkl')

print('Saved to ../models/health_condition_model.pkl')
print('Feature columns:', feature_cols)